In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

# ============================================================
# 1. CONFIGURAÇÕES DA SIMULAÇÃO E POLOS DE INVESTIGAÇÃO
# ============================================================

# Período de amostragem nominal (exemplo do enunciado: Ts = 0.1 s)
TS_NOMINAL = 0.1

# Conjunto de polos contínuos para investigação (Plano-s)
polos_s = [
    -2.0,  # Polo real estável (Exemplo da apostila: s = -2)
    -8.0,  # Polo real rápido
    +1.0,  # Polo real instável
    complex(0, 10.0),  # Par imaginário puro (marginal)
    complex(0, -10.0),
    complex(-1.0, 5.0),  # Par complexo conjugado estável
    complex(-1.0, -5.0),
    complex(+0.5, 4.0),  # Par complexo conjugado instável
    complex(+0.5, -4.0),
]

# Faixa de valores de Ts para analisar o efeito do período de amostragem
lista_Ts = [0.02, 0.05, 0.1, 0.2, 0.5]


# ============================================================
# 2. MODELAGEM MATEMÁTICA MANUAL (MAPEAMENTO s -> z)
# ============================================================


def mapear_s_para_z_manual(s_polo, Ts):
  """Mapeia um polo s = sigma + j*omega do plano-s para o plano-z via z = exp(s * Ts).

  Calcula manualmente:
  - Modulo |z| = exp(sigma * Ts)
  - Angulo theta = omega * Ts (rad)
  - z = exp(sigma*Ts) * [cos(omega*Ts) + j*sin(omega*Ts)]
  """
  sigma = np.real(s_polo)
  omega = np.imag(s_polo)

  modulo_z = np.exp(sigma * Ts)
  angulo_z = omega * Ts

  z_real = modulo_z * np.cos(angulo_z)
  z_imag = modulo_z * np.sin(angulo_z)

  return complex(z_real, z_imag), modulo_z, angulo_z


def classificar_estabilidade_s(s_polo, tol=1e-9):
  """Classifica a estabilidade no plano-s baseada em Re{s}."""
  sigma = np.real(s_polo)
  if sigma < -tol:
    return "ESTÁVEL (Re<0)"
  elif sigma > tol:
    return "INSTÁVEL (Re>0)"
  else:
    return "MARGINAL (Re=0)"


def classificar_estabilidade_z(modulo_z, tol=1e-9):
  """Classifica a estabilidade no plano-z baseada em |z|."""
  if modulo_z < 1.0 - tol:
    return "ESTÁVEL (|z|<1)"
  elif modulo_z > 1.0 + tol:
    return "INSTÁVEL (|z|>1)"
  else:
    return "MARGINAL (|z|=1)"


# ============================================================
# 3. VERIFICAÇÃO COM BIBLIOTECAS (scipy.signal)
# ============================================================


def verificar_discretizacao_scipy(s_polo, Ts):
  """Valida a conversao continuo -> discreto usando c2d (impulso equivalente)."""
  _, poles_d, _, _ = signal.cont2discrete(
      ([], [s_polo], 1.0), Ts, method="impulse"
  )
  return poles_d[0]


# ============================================================
# 4. PAINEL DE VISUALIZAÇÃO 1: MAPEAMENTO LADO A LADO (Plano-s vs Plano-z)
# ============================================================

fig, eixos = plt.subplots(1, 2, figsize=(14, 6))

# --- (A) PLANO-s ---
eixos[0].axvspan(-10, 0, color="tab:green", alpha=0.10, label="Região Estável")
eixos[0].axvspan(0, 4, color="tab:red", alpha=0.10, label="Região Instável")

eixos[0].axhline(0, color="k", linewidth=0.8)
eixos[0].axvline(0, color="k", linewidth=1.2, label=r"Eixo $j\omega$ (Marginal)")

cores = plt.cm.tab10(np.linspace(0, 1, len(polos_s)))

for i, p_s in enumerate(polos_s):
  re_s, im_s = np.real(p_s), np.imag(p_s)
  rotulo = f"s{i+1} = {re_s:+.1f}{im_s:+.1f}j" if im_s != 0 else f"s{i+1} = {re_s:+.1f}"
  eixos[0].plot(
      re_s, im_s, "x", color=cores[i], markersize=10, markeredgewidth=2.5
  )
  eixos[0].annotate(
      rotulo, (re_s, im_s), textcoords="offset points", xytext=(6, 6), fontsize=8
  )

eixos[0].set_xlim(-9, 3)
eixos[0].set_ylim(-12, 12)
eixos[0].set_title(r"Plano-$s$ (Sistemas Contínuos)")
eixos[0].set_xlabel(r"Re(s) = $\sigma$")
eixos[0].set_ylabel(r"Im(s) = $j\omega$")
eixos[0].grid(True)
eixos[0].legend(loc="lower left")

# --- (B) PLANO-z ---
theta_c = np.linspace(0, 2 * np.pi, 300)
eixos[1].plot(
    np.cos(theta_c),
    np.sin(theta_c),
    "k--",
    linewidth=1.5,
    label="Círculo Unitário (|z| = 1)",
)
circ_int = plt.Circle(
    (0, 0), 1.0, color="tab:green", alpha=0.10, label="Região Estável"
)
eixos[1].add_patch(circ_int)
eixos[1].axvspan(-2, 2, color="tab:red", alpha=0.05)

eixos[1].axhline(0, color="k", linewidth=0.8)
eixos[1].axvline(0, color="k", linewidth=0.8)

for i, p_s in enumerate(polos_s):
  z_m, mod_z, ang_z = mapear_s_para_z_manual(p_s, TS_NOMINAL)
  re_z, im_z = np.real(z_m), np.imag(z_m)
  rotulo = (
      f"z{i+1} = {mod_z:.2f}∠{np.degrees(ang_z):.0f}°"
      if abs(im_z) > 1e-6
      else f"z{i+1} = {re_z:.3f}"
  )

  eixos[1].plot(
      re_z, im_z, "x", color=cores[i], markersize=10, markeredgewidth=2.5
  )
  eixos[1].annotate(
      rotulo, (re_z, im_z), textcoords="offset points", xytext=(6, 6), fontsize=8
  )

eixos[1].set_xlim(-1.6, 1.6)
eixos[1].set_ylim(-1.6, 1.6)
eixos[1].set_aspect("equal", "box")
eixos[1].set_title(
    rf"Plano-$z$ Mapeado via $z = e^{{s T_s}}$ ($T_s = {TS_NOMINAL}\text{{ s}}$)"
)
eixos[1].set_xlabel("Re(z)")
eixos[1].set_ylabel("Im(z)")
eixos[1].grid(True)
eixos[1].legend(loc="upper left")

plt.suptitle(
    r"Mapeamento Fundamental $z = e^{s T_s} = e^{\sigma T_s} e^{j\omega T_s}$",
    fontsize=13,
)
plt.tight_layout()
plt.show()


# ============================================================
# 5. PAINEL DE VISUALIZAÇÃO 2: EFEITO DA VARIAÇÃO DO TEMPO DE AMOSTRAGEM (Ts)
# ============================================================

fig, ax_ts = plt.subplots(figsize=(8, 8))

ax_ts.plot(
    np.cos(theta_c),
    np.sin(theta_c),
    "k--",
    linewidth=1.5,
    label="Círculo Unitário (|z| = 1)",
)
ax_ts.axhline(0, color="k", linewidth=0.8)
ax_ts.axvline(0, color="k", linewidth=0.8)

polo_teste_s = complex(-1.0, 5.0)  # s = -1 ± j5 rad/s
Ts_sweep = np.linspace(0.005, 0.6, 200)

z_trajetoria = [mapear_s_para_z_manual(polo_teste_s, ts)[0] for ts in Ts_sweep]
ax_ts.plot(
    np.real(z_trajetoria),
    np.imag(z_trajetoria),
    "gray",
    linestyle=":",
    linewidth=1.5,
    label="Trajetória contínua com Ts",
)

estilos_marcadores = ["o", "s", "^", "D", "*"]
for idx, ts_val in enumerate(lista_Ts):
  z_p, mod_p, ang_p = mapear_s_para_z_manual(polo_teste_s, ts_val)
  ax_ts.plot(
      np.real(z_p),
      np.imag(z_p),
      marker=estilos_marcadores[idx],
      color=f"C{idx}",
      markersize=9,
      label=f"Ts = {ts_val:g} s  (|z|={mod_p:.3f}, θ={np.degrees(ang_p):.0f}°)",
  )

ax_ts.set_xlim(-1.3, 1.3)
ax_ts.set_ylim(-1.3, 1.3)
ax_ts.set_aspect("equal", "box")
ax_ts.set_title(
    r"Efeito da Variação de $T_s$ na Posição do Polo Mapeado ($s = -1 + j5$)"
)
ax_ts.set_xlabel("Re(z)")
ax_ts.set_ylabel("Im(z)")
ax_ts.legend(loc="upper right")
ax_ts.grid(True)

plt.tight_layout()
plt.show()


# ============================================================
# 6. PAINEL DE VISUALIZAÇÃO 3: RESPOSTA TEMPORAL CONTINUA x DISCRETA
# ============================================================

# Polo do exemplo do roteiro: s = -2, Ts = 0.1 s => z = exp(-0.2) ≈ 0.819
s_exemplo = -2.0
z_exemplo, _, _ = mapear_s_para_z_manual(s_exemplo, TS_NOMINAL)

t_cont = np.linspace(0, 2.5, 500)
n_disc = np.arange(int(2.5 / TS_NOMINAL))
t_disc = n_disc * TS_NOMINAL

# Resposta ao impulso contínua h(t) = exp(-2t) e discreta h[n] = (exp(-0.2))^n
h_cont = np.exp(s_exemplo * t_cont)
h_disc = (np.real(z_exemplo)) ** n_disc

# Resposta ao degrau contínua y(t) = 0.5*(1 - exp(-2t)) e discreta equivalente
y_cont = (1.0 - np.exp(s_exemplo * t_cont)) / abs(s_exemplo)
y_disc = np.cumsum(h_disc) * TS_NOMINAL

fig, eixos_t = plt.subplots(1, 2, figsize=(13, 5))

# (A) Resposta ao Impulso
eixos_t[0].plot(
    t_cont, h_cont, "C0-", linewidth=2, label=r"Contínuo $h(t) = e^{-2t}$"
)
eixos_t[0].stem(
    t_disc,
    h_disc,
    linefmt="C1--",
    markerfmt="C1o",
    label=rf"Discreto $h[n] = ({np.real(z_exemplo):.3f})^n$",
)
eixos_t[0].set_title(r"Resposta ao Impulso ($s = -2 \rightarrow z \approx 0.819$)")
eixos_t[0].set_xlabel("Tempo (s)")
eixos_t[0].set_ylabel("Amplitude")
eixos_t[0].legend()
eixos_t[0].grid(True)

# (B) Resposta ao Degrau
eixos_t[1].plot(t_cont, y_cont, "C0-", linewidth=2, label=r"Contínuo $y(t)$")
eixos_t[1].stem(
    t_disc,
    y_disc,
    linefmt="C1--",
    markerfmt="C1s",
    label=r"Discreto $y[n]$ Amostrado",
)
eixos_t[1].set_title(rf"Resposta ao Degrau ($T_s = {TS_NOMINAL}\text{{ s}}$)")
eixos_t[1].set_xlabel("Tempo (s)")
eixos_t[1].set_ylabel("Amplitude")
eixos_t[1].legend()
eixos_t[1].grid(True)

plt.tight_layout()
plt.show()


# ============================================================
# 7. ANÁLISE NUMÉRICA E RELATÓRIO NO TERMINAL
# ============================================================

print("TÓPICO 18 - RELAÇÃO ENTRE LAPLACE (PLANO-s) E TRANSFORMADA-Z (PLANO-z)")
print("=" * 110)
print(f"Período de amostragem nominal: Ts = {TS_NOMINAL} s\n")
print(
    f"{'Polo s (Contínuo)':<22}{'Estabilidade (s)':<18}{'Polo z = exp(s*Ts)':<28}{'|z|':>8}{'θ (rad)':>9}{'Estabilidade (z)':<18}"
)
print("-" * 110)

for p_s in polos_s:
  z_m, mod_z, ang_z = mapear_s_para_z_manual(p_s, TS_NOMINAL)
  est_s = classificar_estabilidade_s(p_s)
  est_z = classificar_estabilidade_z(mod_z)

  re_s, im_s = np.real(p_s), np.imag(p_s)
  txt_s = f"{re_s:+.1f}{im_s:+.1f}j" if im_s != 0 else f"{re_s:+.1f}"

  re_z, im_z = np.real(z_m), np.imag(z_m)
  txt_z = f"{re_z:+.4f}{im_z:+.4f}j" if abs(im_z) > 1e-6 else f"{re_z:+.4f}"

  print(
      f"{txt_s:<22}{est_s:<18}{txt_z:<28}{mod_z:8.4f}{ang_z:9.4f}{est_z:<18}"
  )


print("\n\nVERIFICAÇÃO DO EXEMPLO DA APOSTILA (s = -2, Ts = 0.1 s):")
print("=" * 70)
z_ex_manual, mod_ex, _ = mapear_s_para_z_manual(-2.0, 0.1)
z_ex_scipy = verificar_discretizacao_scipy(-2.0, 0.1)
erro_ex = abs(z_ex_manual - z_ex_scipy)

print(f"  - Valor Teórico Analítico : z = e^(-0.2) = {np.exp(-0.2):.7f}")
print(f"  - Valor Calculado Manual  : z = {z_ex_manual.real:.7f}")
print(f"  - Valor SciPy (c2d)       : z = {z_ex_scipy.real:.7f}")
print(f"  - Erro Absoluto           : {erro_ex:.3e}")
print(f"  - Condição de Estabilidade: |z| = {mod_ex:.4f} < 1 (SISTEMA ESTÁVEL)")


print("\n\nANÁLISE DO EFEITO DA VARIAÇÃO DO PERÍODO DE AMOSTRAGEM (Ts):")
print("Polo s = -1.0 + 5.0j rad/s")
print("=" * 75)
print(
    f"{'Ts (s)':<10}{'Polo z Mapeado':<30}{'Módulo |z|':>12}{'Ângulo θ (°)':>15}{'Frequência Discreta':>18}"
)
print("-" * 75)

for ts_val in lista_Ts:
  z_p, mod_p, ang_p = mapear_s_para_z_manual(complex(-1.0, 5.0), ts_val)
  print(
      f"{ts_val:<10.2f}{f'{z_p.real:+.4f}{z_p.imag:+.4f}j':<30}{mod_p:12.4f}{np.degrees(ang_p):15.1f}°{ang_p:18.4f}"
  )


print("\n\nCONCLUSÕES TEÓRICAS DA INVESTIGAÇÃO (RESPOSTAS AOS TÓPICOS):")
print("=" * 80)
print(
    "1. Equação de Mapeamento Fundamental:\n"
    "   z = exp(s * Ts) = exp(sigma * Ts) * exp(j * omega * Ts)\n"
    "   - O módulo do polo discreto é dadas exclusivamente por |z| = exp(sigma *"
    " Ts).\n"
    "   - O ângulo (frequência discreta) é dado por theta = omega * Ts"
    " (rad/amostra).\n"
)
print(
    "2. Conservação da Estabilidade (Mapeamento de Regiões):\n"
    "   - Semiplano Esquerdo (Re{s} < 0)  --> Mapeia para DENTRO do círculo"
    " unitário (|z| < 1) -> ESTÁVEL.\n"
    "   - Eixo Imaginário (Re{s} = 0)     --> Mapeia EXATAMENTE sobre o"
    " círculo unitário (|z| = 1) -> MARGINAL.\n"
    "   - Semiplano Direito (Re{s} > 0)   --> Mapeia para FORA do círculo"
    " unitário (|z| > 1) -> INSTÁVEL.\n"
)
print(
    "3. Efeito do Período de Amostragem (Ts):\n"
    "   - Reduzir Ts (Amostragem Rápida)   --> Traz os polos no plano-z para"
    " perto de z = +1.\n"
    "   - Aumentar Ts (Amostragem Lenta)    --> Aumenta a rotação angular"
    " (theta = omega * Ts).\n"
    "     Se theta > pi (ou seja, f_amostragem < 2 * f_sinal), ocorre ALIASING"
    " e sobreposição de polos.\n"
)

print("Simulação do Tópico 18 concluída com sucesso!")